# 🧠 Beyin Tümörü Sınıflandırması - Google Colab Eğitim Notebook'u

Bu notebook, RTX 3050 gibi düşük VRAM'li kartlarda uzun sürebilecek eğitimi
Colab'ın ücretsiz T4 GPU'sunda çalıştırmak için hazırlanmıştır.

**Önce yapman gerekenler:**
1. Üstteki menüden `Çalışma Zamanı > Çalışma zamanı türünü değiştir > T4 GPU` seç.
2. Kaggle hesabından `kaggle.json` dosyanı indir (Kaggle > Settings > API > Create New Token).
3. Aşağıdaki hücreleri sırayla çalıştır; Kaggle dosyasını yükleman istenecek.

In [18]:
%cd /content
!rm -rf brain-tumor-classification
!git clone https://github.com/emirhanuzen/brain-tumor-classification.git
%cd brain-tumor-classification
!pip install -q -r requirements.txt

/content
Cloning into 'brain-tumor-classification'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 7), reused 25 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 10.85 KiB | 10.85 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/brain-tumor-classification


## 1. GPU kontrolü

In [1]:
import torch
print("CUDA kullanılabilir mi:", torch.cuda.is_available())
print("Cihaz:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA kullanılabilir mi: True
Cihaz: Tesla T4


## 2. GitHub reposunu klonla

In [11]:
!rm -rf brain-tumor-classification
!git clone https://github.com/emirhanuzen/brain-tumor-classification.git
%cd brain-tumor-classification
!pip install -q -r requirements.txt

Cloning into 'brain-tumor-classification'...
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 25 (delta 3), reused 23 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (25/25), 10.51 KiB | 10.51 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/brain-tumor-classification/brain-tumor-classification


## 3. Kaggle API kurulumu
Aşağıdaki hücreyi çalıştırınca dosya seçme penceresi açılacak, `kaggle.json`'ı seç.

In [14]:
from google.colab import files
uploaded = files.upload()  # kaggle.json seç

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


## 4. Veri setini indir

In [15]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p data/ --unzip
!ls data/Training

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
100% 157M/157M [00:00<00:00, 223MB/s]

glioma	meningioma  notumor  pituitary


## 5. Eğitimi başlat
T4 GPU ile 15 epoch genelde 10-20 dakika sürer (dataset boyutuna göre değişir).

In [16]:
%cd src
!python train.py --epochs 15 --batch-size 32 --lr 0.0001 --data-dir ../data
%cd ..

/content/brain-tumor-classification/src
Kullanılan cihaz: cuda
Epoch 1/15: 100% 175/175 [00:32<00:00,  5.38it/s]
Epoch 1: train_loss=1.0154 val_loss=0.9627 val_acc=0.5150
  → Yeni en iyi model kaydedildi (val_acc=0.5150)
Epoch 2/15: 100% 175/175 [00:31<00:00,  5.54it/s]
Epoch 2: train_loss=0.9277 val_loss=0.8815 val_acc=0.5919
  → Yeni en iyi model kaydedildi (val_acc=0.5919)
Epoch 3/15: 100% 175/175 [00:29<00:00,  5.83it/s]
Epoch 3: train_loss=0.8485 val_loss=0.8097 val_acc=0.7025
  → Yeni en iyi model kaydedildi (val_acc=0.7025)
Epoch 4/15: 100% 175/175 [00:30<00:00,  5.79it/s]
Epoch 4: train_loss=0.7723 val_loss=0.7420 val_acc=0.7506
  → Yeni en iyi model kaydedildi (val_acc=0.7506)
Epoch 5/15: 100% 175/175 [00:30<00:00,  5.81it/s]
Epoch 5: train_loss=0.7034 val_loss=0.6823 val_acc=0.7819
  → Yeni en iyi model kaydedildi (val_acc=0.7819)
Epoch 6/15: 100% 175/175 [00:31<00:00,  5.48it/s]
Epoch 6: train_loss=0.6488 val_loss=0.6423 val_acc=0.7919
  → Yeni en iyi model kaydedildi (val_a

## 6. Değerlendirme (Confusion Matrix + Rapor)

In [17]:
%cd src
!python evaluate.py --model-path models/best_model.pth --output-dir ../outputs --data-dir ../data
%cd ..

/content/brain-tumor-classification/src
              precision    recall  f1-score   support

   Malignant       0.86      0.58      0.69       400
      Benign       0.84      0.89      0.86       800
    No Tumor       0.79      0.97      0.87       400

    accuracy                           0.83      1600
   macro avg       0.83      0.81      0.81      1600
weighted avg       0.83      0.83      0.82      1600

Confusion matrix kaydedildi: ../outputs/confusion_matrix.png
/content/brain-tumor-classification


## 7. Model ve sonuçları indir
Eğitilen modeli ve confusion matrix'i bilgisayarına indirip GitHub'a push edebilirsin
(ya da doğrudan Colab'dan Google Drive'a kaydedip oradan indirebilirsin).

In [ ]:
from google.colab import files
files.download("models/best_model.pth")
files.download("outputs/confusion_matrix.png")